# 02 - Baseline: TF-IDF + Logistic Regression

| Step | Details |
|---|---|
| Validation | fixed split from `common.split_train_val` |
| Features | TF-IDF word 1-2 / char_wb 2-5 n-grams |
| Model | LogisticRegression, grid over `C` by macro-F1 |
| Outputs | per-class report, confusion matrix, `submission_baseline.csv`, probabilities in `outputs/` |

- Vectorizers are fitted **on the train part only** (no leakage).
- Shared code: `src/common.py`.

## Setup

In [ ]:
import subprocess
import sys

REPO = "https://github.com/mohinska/ai_hw3_AIGenTextDetection.git"
REPO_DIR = "/tmp/repo"

subprocess.run(["rm", "-rf", REPO_DIR], check=True)
subprocess.run(["git", "clone", "-q", "--depth", "1", REPO, REPO_DIR], check=True)
sys.path.insert(0, f"{REPO_DIR}/src")

In [ ]:
import os
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, f1_score

import common
from common import LABELS, TEXT_COL, LABEL_COL, SEED
from features import TFIDF, fit_tfidf, tfidf_transform

warnings.filterwarnings("ignore", category=UserWarning)

## Config

In [ ]:
N_TRAIN = 60_000         # stratified subsample of the train part used for fitting (None = all)
REFIT_ON_TRAIN_PLUS_VAL = False  # True: refit best config on train+val before predicting test

OUT_DIR = os.path.join(common.WORK_DIR, "outputs")
SUBMISSION_PATH = os.path.join(common.WORK_DIR, "submission_baseline.csv")
os.makedirs(OUT_DIR, exist_ok=True)

## Load data

In [ ]:
train_df, test_df, sample_sub = common.load_data()
LANG_COL = common.lang_col(train_df)

assert set(train_df[LABEL_COL].unique()) == set(LABELS), train_df[LABEL_COL].unique()
print("train:", train_df.shape, "| test:", test_df.shape, "| columns:", list(train_df.columns))
train_df[LABEL_COL].value_counts(normalize=True).round(3)

## Fixed validation split

| Parameter | Value |
|---|---|
| Size | 10% of train.csv |
| Stratified by | label + language |
| Seed | 42 |


In [ ]:
trn, val = common.split_train_val(train_df)
val[["id"]].to_csv(os.path.join(OUT_DIR, f"val_ids_seed{SEED}.csv"), index=False)
trn = common.subsample(trn, N_TRAIN)
print(f"train part: {len(trn):,} | val: {len(val):,}")

In [ ]:
def prep(texts):
    return texts.fillna("").astype(str)

X_trn_txt, y_trn = prep(trn[TEXT_COL]), trn[LABEL_COL].values
X_val_txt, y_val = prep(val[TEXT_COL]), val[LABEL_COL].values
X_test_txt = prep(test_df[TEXT_COL])

## Parameter search

- Each vectorizer is fitted once on the train part; validation is only transformed.
- Feature sets: word, char, word+char.
- Grid: `C` in `C_GRID`, metric: macro-F1.

In [ ]:
C_GRID = [0.5, 2.0, 8.0]


fit_vectorizers, transform = fit_tfidf, tfidf_transform   # src/features.py (shared with 04 / 06)


def make_clf(C):
    return LogisticRegression(C=C, max_iter=2000, random_state=SEED)


feature_sets = {"word_1-2": ["word_1-2"],
                "char_wb_2-5": ["char_wb_2-5"],
                "word+char": ["word_1-2", "char_wb_2-5"]}

fitted = fit_vectorizers(TFIDF.keys(), X_trn_txt)   # fit on train part only
results = []
for fs_name, vec_names in feature_sets.items():
    vecs = {n: fitted[n] for n in vec_names}
    Xtr, Xva = transform(vecs, X_trn_txt), transform(vecs, X_val_txt)
    for C in C_GRID:
        t0 = time.time()
        clf = make_clf(C).fit(Xtr, y_trn)
        f1 = f1_score(y_val, clf.predict(Xva), average="macro")
        results.append({"features": fs_name, "C": C, "n_feats": Xtr.shape[1],
                        "macro_f1": round(f1, 4), "fit_s": round(time.time() - t0, 1)})

results_df = pd.DataFrame(results).sort_values("macro_f1", ascending=False).reset_index(drop=True)
results_df.to_csv(os.path.join(OUT_DIR, "baseline_grid.csv"), index=False)
results_df

## Best model on the validation split

In [ ]:
best = results_df.iloc[0]
best_vecs = {n: fitted[n] for n in feature_sets[best["features"]]}
Xtr, Xva = transform(best_vecs, X_trn_txt), transform(best_vecs, X_val_txt)
best_clf = make_clf(best["C"]).fit(Xtr, y_trn)

val_proba = best_clf.predict_proba(Xva)
val_pred = best_clf.classes_[val_proba.argmax(1)]
print(f"best: {best['features']}, C={best['C']} -> val macro-F1 = {f1_score(y_val, val_pred, average='macro'):.4f}\n")
print(classification_report(y_val, val_pred, labels=LABELS, digits=4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, norm, title in zip(axes, [None, "true"], ["counts", "row-normalised"]):
    ConfusionMatrixDisplay.from_predictions(y_val, val_pred, labels=LABELS, normalize=norm,
                                            values_format=".2f" if norm else "d",
                                            cmap="Blues", colorbar=False, ax=ax)
    ax.set_title(f"Baseline confusion matrix ({title})")
common.save_fig("baseline_cm.png")

In [ ]:
# macro-F1 per language and per domain: where the model is weak
for col in [LANG_COL, "domain"]:
    if col in val.columns:
        display(common.f1_by(y_val, val_pred, val[col]))

In [ ]:
# Save validation probabilities: reused later for ensembling / threshold tuning
common.save_proba(val["id"], val_proba, best_clf.classes_, "baseline", "val")

## Predict test and write submission

In [ ]:
if REFIT_ON_TRAIN_PLUS_VAL:
    full_txt = pd.concat([X_trn_txt, X_val_txt], ignore_index=True)
    full_y = np.concatenate([y_trn, y_val])
    final_vecs = fit_vectorizers(feature_sets[best["features"]], full_txt)
    final_clf = make_clf(best["C"]).fit(transform(final_vecs, full_txt), full_y)
else:
    final_vecs, final_clf = best_vecs, best_clf

test_proba = final_clf.predict_proba(transform(final_vecs, X_test_txt))
test_pred = final_clf.classes_[test_proba.argmax(1)]

common.save_proba(test_df["id"], test_proba, final_clf.classes_, "baseline", "test")

submission = common.save_submission(test_df["id"], test_pred, sample_sub, SUBMISSION_PATH)
print("saved", SUBMISSION_PATH, submission.shape)
submission["label"].value_counts(normalize=True).round(3)

## Conclusions

| Finding | Decision |
|---|---|
| <!-- TODO --> | <!-- TODO --> |
| <!-- TODO --> | <!-- TODO --> |